In [0]:
%sql
CREATE OR REPLACE TABLE workspace.climate_gold.district_monthly
COMMENT 'Monthly weather aggregates per sub-county'
AS
SELECT district_id,
       CAST(date_trunc('month', date) AS DATE)       AS month,
       ROUND(SUM(precip_mm), 1)                      AS rain_mm,
       ROUND(AVG(root_soil_wetness), 3)              AS soil_wetness,
       ROUND(AVG(tmax_c), 2)                         AS tmax_avg_c,
       SUM(CASE WHEN tmax_c > 35 THEN 1 ELSE 0 END)  AS heat_days,
       COUNT(*)                                      AS days_observed
FROM workspace.climate_silver.weather_daily
GROUP BY district_id, CAST(date_trunc('month', date) AS DATE);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.climate_gold.district_anomalies
COMMENT '3-month rolling conditions compared with each sub-county 1991-2020 normal for the same time of year'
AS
WITH roll AS (
  SELECT *,
    SUM(rain_mm)      OVER w AS rain_3m,
    AVG(soil_wetness) OVER w AS soil_3m,
    AVG(tmax_avg_c)   OVER w AS tmax_3m,
    COUNT(*)          OVER w AS months_in_window
  FROM workspace.climate_gold.district_monthly
  WINDOW w AS (PARTITION BY district_id ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
),
baseline AS (
  SELECT district_id,
         month(month)  AS cal_month,
         AVG(rain_3m)  AS rain_mu,  STDDEV(rain_3m) AS rain_sd,
         AVG(soil_3m)  AS soil_mu,  STDDEV(soil_3m) AS soil_sd,
         AVG(tmax_3m)  AS tmax_mu,  STDDEV(tmax_3m) AS tmax_sd
  FROM roll
  WHERE year(month) BETWEEN 1991 AND 2020
  GROUP BY district_id, month(month)
)
SELECT r.district_id,
       r.month,
       ROUND(r.rain_3m, 1)                                        AS rain_3m_mm,
       ROUND(b.rain_mu, 1)                                        AS rain_3m_normal_mm,
       ROUND(100 * r.rain_3m / NULLIF(b.rain_mu, 0), 0)           AS rain_pct_of_normal,
       ROUND((r.rain_3m - b.rain_mu) / NULLIF(b.rain_sd, 0), 2)   AS rain_z,
       ROUND((r.soil_3m - b.soil_mu) / NULLIF(b.soil_sd, 0), 2)   AS soil_z,
       ROUND((r.tmax_3m - b.tmax_mu) / NULLIF(b.tmax_sd, 0), 2)   AS heat_z
FROM roll r
JOIN baseline b
  ON r.district_id = b.district_id AND month(r.month) = b.cal_month
WHERE r.months_in_window = 3;

In [0]:
%sql
SELECT COUNT(*)                     AS rows,
       COUNT(DISTINCT district_id)  AS districts,
       MIN(month)                   AS first_month,
       MAX(month)                   AS last_month,
       ROUND(AVG(CASE WHEN year(month) BETWEEN 1991 AND 2020 THEN rain_z END), 3) AS baseline_avg_rain_z
FROM workspace.climate_gold.district_anomalies;